# 02 – Characters

Esplorazione e data cleaning del dataset `characters.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `character_mal_id` | ID univoco del personaggio su MyAnimeList |
| `url` | URL della pagina MAL del personaggio |
| `name` | Nome del personaggio |
| `name_kanji` | Nome in giapponese |
| `image` | URL dell'immagine del personaggio |
| `favorites` | Numero di utenti che hanno aggiunto il personaggio ai preferiti |
| `about` | Descrizione testuale del personaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

df_characters = pd.read_csv('../datasets/characters.csv')
print(f'Shape: {df_characters.shape}')
print()
df_characters.info()
df_characters.head()

Shape: (209963, 7)

<class 'pandas.DataFrame'>
RangeIndex: 209963 entries, 0 to 209962
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   character_mal_id  209961 non-null  float64
 1   url               209961 non-null  str    
 2   name              209961 non-null  str    
 3   name_kanji        154483 non-null  str    
 4   image             209961 non-null  str    
 5   favorites         209961 non-null  float64
 6   about             112987 non-null  str    
dtypes: float64(2), str(5)
memory usage: 11.2 MB


,character_mal_id,url,name,name_kanji,image,favorites,about
0,280386.0,https://myanimelist.net/character/280386/Envi_...,Envi Mel Champagne,エンヴィ・メル・シャンパーニュ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
1,280354.0,https://myanimelist.net/character/280354/Eleven,Eleven,イレヴン,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
2,280353.0,https://myanimelist.net/character/280353/Stud,Stud,スタッド,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
3,280352.0,https://myanimelist.net/character/280352/Judge,Judge,ジャッジ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
4,280339.0,https://myanimelist.net/character/280339/Eiji_...,Eiji Kurokawa,黒川 英治,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0,NaN


Il dataset contiene **209.963** righe che corispondono ai personaggi e **7** colonne che corrispondo agli attributi per ciascuno. 

Notiamo che il numero di righe non nulle non corrisponde al numero totale di righe, il che significa che ci sono dei valori mancanti da gestire. 

Notiamo anche che `character_mal_id` e `favorite` sono di tipo `float64` invece di `int` il che potrebbe portare a errori, occupa più memoria del necessario e per numeri molto grandi potrebbe ridurre la leggibilità. 

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [2]:
n_originale = len(df_characters) #calcoliamo il numero totale di righe

mask_dup = df_characters.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_characters[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_characters.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_characters):,}')

Righe totali coinvolte in duplicazioni : 796
  → prime occorrenze mantenute         : 340
  → occorrenze extra rimosse           : 456

Righe prima della rimozione : 209,963
Righe dopo la rimozione     : 209,507


Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `character_mal_id`
 Questa è la chiave primaria del dataset: ogni riga deve avere un ID univoco e non nullo.

In [3]:
analyze(df_characters['character_mal_id'])


  Nome serie                     character_mal_id
  dtype                          float64
  Dimensione                     209,507
  Range indice                   0 … 209962
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,507
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     1  (0.00%)
  Zeri                           0  (0.00%)
  Positivi                       209,506   (100.00%)
  Negativi                       0   (0.00%)
  Valori unici                   209,506  (100.00%)
  Valori float                   0

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            1.0000
  Max                            282,284.00
  Range                          282,283.00
  Media                          150,380.10
  Mediana                        159

**Osservazioni:**
- La colonna è `float64` invece di `int`, probabilmente dovuto alla presenza di valori null. Questo è un problema perché occupa più memoria del necessario e può ridurre la leggibilità per numeri grandi. Abbiamo verificato che non esistono valori di tipo `float`.
- Su 209.507 righe, c'è esattamente 1 valore null. Essendo la chiave primaria, non può essere tollerato e quindi quella riga va rimossa.  
- Tutti i 209.506 valori non nulli sono unici (100% unicità), coerente con il ruolo di chiave primaria. I duplicati sono già stati eliminati in precedenza.     
- Non ci sono outliers. Con IQR = 150.688, le soglie outlier sarebbero -151.636 e 451.116. I nostri ID vanno da 1 a 282.284. Entrambi rientrano dentro le soglie.

**Pulizie necessarie:**
- Convertiamo i valori di tipo `float64` a `int`. 
- Rimuoviamo le righe con `character_mal_id` null in quanto è chiave primaria. 
- Non ci sono duplicati da rimuovere in quanto sono già stati gestiti durante la   
  rimozione dei duplicati esatti.

In [4]:
df_characters.dropna(subset=['character_mal_id'], inplace=True)
non_interi = (df_characters['character_mal_id'] % 1 != 0).sum()
print(f'Valori non interi: {non_interi}')
df_characters['character_mal_id'] = df_characters['character_mal_id'].astype(int)
print(f'Righe dopo pulizia character_mal_id: {len(df_characters):,}')

Valori non interi: 0
Righe dopo pulizia character_mal_id: 209,506


### 2.2 `url`
Questa colonna contiene il link alla pagina del personaggio su MyAnimeList. Utilizziamo `strip()` per rimuovere eventuali spazzi vuoti all inizio e fine del URL. Effetuiamo poi l'analisi usando la nostra libreria `dataset_analyzer` e facciamo un controllo sul format del URL. Controlliamo anche la consistenza `character_mal_id` ↔ `url`. L'URL contiene l'ID del personaggio nel percorso (`/character/{id}/`). Verifichiamo che l'ID estratto dall'URL corrisponda al valore di `character_mal_id` per ogni riga.

In [5]:
df_characters['url'] = df_characters['url'].str.strip()
analyze(df_characters['url'])

pattern = r'^https://myanimelist\.net/character/\d+/\S+$'                                                               
non_conformi = df_characters[~df_characters['url'].str.match(pattern)]                                                  
print(f'URL non conformi al formato atteso: {len(non_conformi)}') 
if len(non_conformi) > 0:
    print(non_conformi[['character_mal_id', 'url', 'name']].to_string(index=True))

id_from_url = df_characters['url'].str.extract(r'/character/(\d+)/')[0].astype(int)
mismatch = (id_from_url != df_characters['character_mal_id'])
print()
print(f'Righe con ID non coerente tra url e character_mal_id: {mismatch.sum()}')


  Nome serie                     url
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   209,506  (100.00%)
  Valori duplicati               0  (0.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  39  caratteri
  Lunghezza max                  94  caratteri
  Lunghezza media                51.19  caratteri
  Lunghezza mediana              52.0000  caratteri
  Dev. standard lunghezza        4.59
  IQR lunghezza                  7.0000

Valori di lungh


**Osservazioni**
- Non ci sono dei valori nulli o duplicati. 
- Tutti i 209.506 URL sono unici (100%). 
- Lunghezza varia tra 39 e 94 caratteri. La variazione è interamente dovuta alla lunghezza del nome in coda all'URL. 
- Nessuna inconsistenza `character_id` - `URL` trovata: l'ID estratto dall'URL corrisponde esattamente a `character_mal_id` per tutte le 209.506 righe.

**Nessuna pulizia necessaria.**  

### 2.3 `name`

Nome del personaggio. A differenza di `character_mal_id`, i duplicati sono attesi in quanto personaggi diversi possono condividere lo stesso nome.
Applichiamo `str.strip()` per rimuovere eventuali spazi o caratteri invisibili a inizio e fine stringa, come già fatto per `url`.

In [6]:
df_characters['name'] = df_characters['name'].str.strip()
analyze(df_characters['name'])


  Nome serie                     name
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   161,866  (77.26%)
  Valori duplicati               47,640  (22.74%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  54  caratteri
  Lunghezza media                10.57  caratteri
  Lunghezza mediana              11.0000  caratteri
  Dev. standard lunghezza        4.60
  IQR lunghezza                  8.0000

Valori di 

**Osservazioni**
- Non ci sono stringhe null o vuote. 
- Ci sono 46.719 duplicati (22.3%) che sono attesi e non problematici.
- Ci sono solo 162.787 nomi unici su 209.506 che conferma che i nomi non sono identificatori univoci.     
- Lunghezza minima è di 1 carattere. Lunghezza massima è di 54 caratteri. Effetuiamo un controllo più approfondito.
- Nomi contenenti `@`: sono solo 13 quindi possiamo verificare se si tratta di nomi legittimi o dati corrotti.
- Nomi solo numerici: verifichiamo se si tratta di designazioni valide o placeholder.

In [7]:
cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
lunghezze = df_characters['name'].dropna().str.len()

# Nomi più corti
idx_corti = lunghezze.nsmallest(10).index
print(f"10 nomi più corti:")
print(df_characters.loc[idx_corti].drop(columns=cols_to_drop).to_string())

#Nomi più lunghi
idx_lunghi = lunghezze.nlargest(10).index
print(f"\n10 nomi più lunghi:")
print(df_characters.loc[idx_lunghi].drop(columns=cols_to_drop).to_string())
print() 

# Nomi contenenti '@'
nomi_con_at = df_characters[df_characters['name'].str.contains('@|＠', regex=True)]
print(f"Nomi contenenti '@': {len(nomi_con_at)}")
if len(nomi_con_at) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nomi_con_at.drop(columns=cols_to_drop).to_string())

# Nomi solo numerici
nomi_numerici = df_characters[df_characters['name'].str.fullmatch(r'\d+')]
print(f'\nNomi solo numerici: {len(nomi_numerici)}')
if len(nomi_numerici) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nomi_numerici.drop(columns=cols_to_drop).to_string())

10 nomi più corti:
       character_mal_id                                         url name name_kanji
626              279439  https://myanimelist.net/character/279439/I    I         ぼく
4303             275539  https://myanimelist.net/character/275539/μ    μ        NaN
5718             274018  https://myanimelist.net/character/274018/V    V        NaN
5740             273995  https://myanimelist.net/character/273995/I    I        NaN
11536            267578  https://myanimelist.net/character/267578/A    A        NaN
14545            264455  https://myanimelist.net/character/264455/B    B        NaN
14556            264444  https://myanimelist.net/character/264444/A    A        NaN
22670            255847  https://myanimelist.net/character/255847/X    X          X
22922            255585  https://myanimelist.net/character/255585/N    N        NaN
22927            255580  https://myanimelist.net/character/255580/Y    Y        NaN

10 nomi più lunghi:
        character_mal_id            

- **Nomi di lunghezze estreme**: Tutti i valori estremi corrispondono ai nomi corretti. 
- **Nomi contenenti `@`**: Dall'ispezione si tratta di un separatore usato per distinguere il nome del personaggio dal gruppo/ruolo di appartenenza (es. `Doris@Doll`, `Joe@Admin`, `Devo@Leukocyte`). L'unica eccezione è il personaggio `@Gou` ma anche in questo caso, dalle verifiche risulta che il nome è corretto. Non sono errori.
- **Nomi solo numerici**: sono presenti 26 nomi composti esclusivamente da cifre (es. `777`, `07`, `004989`). In contesti anime è comune che personaggi abbiano designazioni numeriche come nome ufficiale. Dall'ispezione sembrano essere nomi validi. 

**Nessuna pulizia necessaria.**

### 2.4 `name_kanji`
Nome del personaggio in caratteri giapponesi. Applichiamo `str.strip()` per rimuovere eventuali spazi o caratteri invisibili a inizio e fine stringa.  

In [8]:
df_characters['name_kanji'] = df_characters['name_kanji'].str.strip()
analyze(df_characters['name_kanji'])


  Nome serie                     name_kanji
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               154,136  (73.57%)
  Null / NaN                     55,370  (26.43%)
    di cui stringhe vuote        2  (convertite a NULL)
  Valori unici                   128,978  (61.56%)
  Valori duplicati               25,158  (12.01%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  47  caratteri
  Lunghezza media                5.16  caratteri
  Lunghezza mediana              5.0000  caratteri
  Dev. standard lunghezza        2.82
  IQR lunghezza                  3.0000

V

**Osservazioni**
- Ci sono 55.368 null (26.4%). Questo è un valore atteso che si può riferire a personaggi occidentali o di origine non giapponese che non hanno un nome kanji.
- Ci sono 24.972 duplicati che come per name, sono attesi.
- Lunghezza minima è di 1 carattere che come per `name`, è valido.              
- Lunghezza massima è di 47 caratteri. Ci sono personaggi con alias multipli. 
- Ci sono dei nomi contenenti il carattere `@` oppure solo caratteri numerici.Da verificare se si tratta di errori. 

In [9]:
# name_kanji contenenti '@'
nk_con_at = df_characters[df_characters['name_kanji'].notna() & df_characters['name_kanji'].str.contains('@|＠', regex=True)]
print(f"name_kanji contenenti '@': {len(nk_con_at)}")
if len(nk_con_at) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nk_con_at.drop(columns=cols_to_drop).to_string())

# name_kanji solo numerici
nk_numerici = df_characters[df_characters['name_kanji'].notna() & df_characters['name_kanji'].str.fullmatch(r'\d+')]
print(f'\nname_kanji solo numerici: {len(nk_numerici)}')
if len(nk_numerici) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nk_numerici.drop(columns=cols_to_drop).to_string())

name_kanji contenenti '@': 11
        character_mal_id                                                                            url                                      name             name_kanji
392               279716                          https://myanimelist.net/character/279716/MioItanGirls                             Mio@ItanGirls              ミオ＠異端ガールズ
34671             242925                           https://myanimelist.net/character/242925/King_T＠KＥD＠                               King T＠KＥD＠             キング・T＠KＥD＠
48518             228277                                   https://myanimelist.net/character/228277/Gou                                      @Gou                     @号
111645            150363                      https://myanimelist.net/character/150363/Himari_Tanahashi                          Himari Tanahashi                    ＠娘々
114837            145663                             https://myanimelist.net/character/145663/DorisDoll                      

**Osservazioni**
- **`name_kanji` contenenti `@`**: Coerentemente con quanto osservato in `name`, il simbolo è usato come separatore nome/gruppo (es. `エルザ@ドール`, `軟体系@アキバ`). Non sono errori.
- **`name_kanji` solo numerici**: sono presenti 7 valori composti solo da cifre (es. `07`, `01`, `245`). Corrispondono ai medesimi personaggi già identificati in `name` con designazione numerica. Non sono errori. 

**Nessuna pulizia necessaria.**

### 2.5 `image`
URL dell'immagine del personaggio su MyAnimeList. Effetuiamo l'analisi usando la nostra libreria `dataset_analyzer` dopo aver usato `strip()` per rimuovere spazi vuoti e facciamo poi un controlo sul format del URL.

In [10]:
df_characters['image'] = df_characters['image'].str.strip()
analyze(df_characters['image'])

pattern_img = r'^https://cdn\.myanimelist\.net/(images/characters/\d+/\d+\.jpg|img/sp/icon/apple-touch-icon-256\.png)$' 
non_conformi_img = df_characters[~df_characters['image'].str.match(pattern_img)]                                        
print(f'Immagini non conformi al formato atteso: {len(non_conformi_img)}')                                              
if len(non_conformi_img) > 0:                                                                                           
    print(non_conformi_img[['character_mal_id', 'image', 'name']].to_string(index=True))



  Nome serie                     image
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   186,327  (88.94%)
  Valori duplicati               23,179  (11.06%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  55  caratteri
  Lunghezza max                  64  caratteri
  Lunghezza media                58.99  caratteri
  Lunghezza mediana              58.0000  caratteri
  Dev. standard lunghezza        1.85
  IQR lunghezza                  1.0000

Valori d

**Osservazione** 
- Non ci sono stringhe vuote o null. Tutta la colonna è popolata. 
- Ci sono 23.179 URL duplicati (11.1%). Tutti corrispondono a un link che porta a un'immagine placeholder. A ogni personaggio senza immagine reale viene assegnato il logo MAL. Non è un errore e non è necessaria nessuna pulizia. 
- Ci sono 186.327 URL unici (88.94%). Tutti i personaggi con immagine reale hanno un URL univoco.
- Tutte le URL seguono il formato corretto senza nessuna anomalia strutturale.
- Nella distribuzione delle lunghezze, le fasce 59–63 hanno 0 occorrenze, il che significa che le URL si dividono in due gruppi: percorso /images/characters/ (55–58 caratteri) e percorso placeholder /img/sp/icon/ (64 caratteri). 


**Nessuna pulizia necessaria**


### 2.6 `favorites`

Numero di utenti che hanno aggiunto il personaggio ai propri preferiti. È una metrica di popolarità.

In [11]:
analyze(df_characters['favorites'])


  Nome serie                     favorites
  dtype                          float64
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           131,681  (62.85%)
  Positivi                       77,825   (37.15%)
  Negativi                       0   (0.00%)
  Valori unici                   2,253  (1.08%)
  Valori float                   0

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0.0000
  Max                            175,632.00
  Range                          175,632.00
  Media                          57.6972
  Mediana                        0.0000
  Mod

**Osservazioni**
- Nessun null, nessun valore negativo. La colonna è completamente popolata e non contiene errori. 
- Il `dtype` è `float64` invece di `int64`, come già visto per `character_mal_id`. 
- Il 62.85% dei valori è 0 (131.681 personaggi). Questo è un valore atteso. Significa che la grande maggioranza dei personaggi non è stata aggiunta nei preferiti come per esemio personaggi di serie nuove.
- Si nota una distribuzione asimetrica con Mediana = 0, Media = 57.7 e Coefficiente di Variazione = 2077%. Significa che pochi personaggi vengo aggiunti di più nei preferiti. 
- P95 = 47 significa che il 95% dei personaggi ha ≤ 47 preferiti.
- Si notano 30,242 outlier secondo IQR. Non sono dati sporchi ma semplicemente il metodo IQR non è adatto a questa colonna. Qualsiasi valore maggiore di 5 viene classificato outlier ma avere più di 5 preferiti non è un anomalia. 
- Ci sono 2.253 valori unici su 209.506 righe (1.08%). Non è un errore in quanto molti personaggi condividono gli stessi conteggi bassi.

**Pulizie necessarie:**
- Convertire il `dtype` float64 a `int`

In [12]:
non_interi = (df_characters['favorites'] % 1 != 0).sum()
print(f'Valori non interi: {non_interi}')
df_characters['favorites'] = df_characters['favorites'].astype(int)
print(f'favorites dtype   : {df_characters["favorites"].dtype}')

Valori non interi: 0
favorites dtype   : int64


In [13]:
cols_to_drop = [c for c in ['image', 'about'] if c in df_characters.columns]

print("10 personaggi con più favorites:")
print(df_characters.nlargest(10, 'favorites').drop(columns=cols_to_drop).to_string())
print()
print("10 personaggi con meno favorites (escludendo 0):")
non_zero = df_characters[df_characters['favorites'] > 0]
print(non_zero.nsmallest(10, 'favorites').drop(columns=cols_to_drop).to_string())

10 personaggi con più favorites:
        character_mal_id                                                       url                name    name_kanji  favorites
208315               417  https://myanimelist.net/character/417/Lelouch_Lamperouge  Lelouch Lamperouge  ルルーシュ・ランペルージ     175632
208690                40       https://myanimelist.net/character/40/Luffy_Monkey_D     Luffy Monkey D.    モンキー・D・ルフィ     145750
168810             45627              https://myanimelist.net/character/45627/Levi                Levi          リヴァイ     144452
208659                71            https://myanimelist.net/character/71/L_Lawliet           L Lawliet      エル ローライト     129333
208668                62         https://myanimelist.net/character/62/Zoro_Roronoa        Zoro Roronoa       ロロノア・ゾロ     114319
208703                27       https://myanimelist.net/character/27/Killua_Zoldyck      Killua Zoldyck    キルア・ゾルディック      99726
208650                80         https://myanimelist.net/character/80/L

### 2.7 `about`
Testo libero con la descrizione del personaggio.

In [14]:
df_characters['about'] = df_characters['about'].str.strip()
analyze(df_characters['about'])


  Nome serie                     about
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               112,736  (53.81%)
  Null / NaN                     96,770  (46.19%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   109,515  (52.27%)
  Valori duplicati               3,221  (1.54%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  3  caratteri
  Lunghezza max                  14,449  caratteri
  Lunghezza media                393.66  caratteri
  Lunghezza mediana              277.0000  caratteri
  Dev. standard lunghezza        440.95
  IQR lunghezza                  317.00

**Osservazioni**
- Notiamo che 46.19% è null (96.770). Questo risultato è atteso in quanto i personaggi secondari o poco conosciuti possono non avere una descrizione.
- Ci sono 3,221 duplicati che controlliamo più in dettaglio successivamente. 
- Tra le stringhe più corte ci sono `"ANN"` e `"None"` che verifichiamo successivamente per determinare se vanno rimosse. 
- Il 87.7% dei valori non-null ha lunghezza ≤ 725 caratteri. La maggior parte delle descrizioni è breve. Solo il 10.6% supera i 725 caratteri. Controlliamo i testi più lunghi per vedere se ci sono anomalie. 
- Risulta 1 URL e 120 valori contenenti `@`. Questo può essere rumore minore, probabilmente link o menzioni. Verifichiamo più in dettaglio. 
- Lingua prevalentemente inglese (parole più comuni: the, to, and, a, of…), ma i 2.314 caratteri unici indicano presenza di testo non-latino (giapponese, arabo, ecc.).

In [15]:
# Valori duplicati
dup_mask = df_characters['about'].duplicated(keep=False) & df_characters['about'].notna()
dup_texts = df_characters.loc[dup_mask, 'about'].value_counts()
print(f'Testi duplicati distinti: {len(dup_texts)}')
print(dup_texts.head(20))

cols_to_drop = [c for c in ['image', 'favorites'] if c in df_characters.columns]

lunghezze = df_characters['about'].dropna().str.len()
print()

# Testi più corti
print("10 testi più corti:")
idx_corti = lunghezze.nsmallest(10).index
print(df_characters.loc[idx_corti].drop(columns=cols_to_drop).to_string())

# Testi più lunghi
print("\n10 testi più lunghi:")
idx_lunghi = lunghezze.nlargest(10).index
print(df_characters.loc[idx_lunghi].drop(columns=cols_to_drop).to_string())

cols_to_drop_about = [c for c in ['image', 'favorites'] if c in df_characters.columns]

# Testi contenenti URL
con_url = df_characters[df_characters['about'].str.contains(r'https?://', regex=True, na=False)]
print(f'Testi contenenti URL: {len(con_url)}')
if len(con_url) > 0:
    print(con_url.drop(columns=cols_to_drop_about).to_string())

# Testi contenenti '@'
con_at = df_characters[df_characters['about'].str.contains('@', regex=False, na=False)]
print(f"\nTesti contenenti '@': {len(con_at)}")
if len(con_at) > 0:
    print(con_at.head(10).drop(columns=cols_to_drop_about).to_string())

Testi duplicati distinti: 1251
about
No voice actors have been added to this character. Help improve our database by searching for a voice actor, and adding this character to their roles .    199
Appears in episode 1.                                                                                                                                       66
Appears in episode 3.                                                                                                                                       59
Appears in episode 4.                                                                                                                                       56
Appears in episode 5.                                                                                                                                       52
Appears in episode 2.                                                                                                                                       50
Appears i

- Ci sono 3.221 duplicati. Da questi, la frase "No voice actors have been added to this character. Help improve our database…" compare 199 volte e rapresenta un placeholder che va rimosso. Abbiamo indagato più in dettaglio e sembra che il resto dei duplicati è composto dalla frase "Appears in episode X.". Questa frase, pur essendo ripetitiva, fornisce un'informazione utile (l'episodio in cui appare il personaggio) e non va rimossa.
-  La stringa più corta è `"ANN"` (3 caratteri): tag residuo senza significato descrittivo, da convertire in `null`. È presente anche `"None"`, anch'essa da convertire in `null`
- I testi contenenti @ e URL sono validi in quanto fanno riferimento a mail oppure fonti di informazioni e dunque vengono tenuti.  

**Pulizia necessaria:** rimozione dei testi boilerplate e dei valori placeholder (`"ANN"`, `"None."`).

In [16]:
placeholder_patterns = [
    r'^No voice actors have been added to this character\.',
    r'^ANN$',
    r'^None\.$',
]
mask_placeholder = df_characters['about'].str.contains(
    '|'.join(placeholder_patterns), regex=True, na=False
)
print(f'Righe con testo placeholder: {mask_placeholder.sum()}')
df_characters.loc[mask_placeholder, 'about'] = np.nan
print(f'Null in about dopo pulizia: {df_characters["about"].isna().sum():,}')

Righe con testo placeholder: 201
Null in about dopo pulizia: 96,971


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effetuiamo il salvataggio del dataset finale.

In [17]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_characters):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_characters):>10,}')

print()
df_characters.to_csv('../datasets_cleaned/characters_clean.csv', index=False)
print('Salvato: datasets_cleaned/characters_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :    209,963
Righe dopo cleaning  :    209,506
Righe rimosse totali :        457

Salvato: datasets_cleaned/characters_clean.csv
